# QAOA improvements: diagnosis, look-ahead QUBO, INTERP

This notebook re-runs every experiment behind the analysis of `starter.ipynb`, so you can see the numbers yourself.

| Section | Question |
|---|---|
| 1–2 | Setup, plus the starter's functions (copied, so this notebook stands alone) |
| 3 | Why does QAOA tie with random guessing? |
| 4 | Why doesn't QAOA reach the MILP's minimum? |
| 5 | Does lowering the penalty raise P(best)? |
| 6 | Look-ahead QUBO term: 7.06% → 1.18% gap |
| 7 | INTERP angle initialization for p > 1 |
| 8 | Generalizing the look-ahead rule, tested on other demand profiles |

**Kernel:** pick **qBraid-SDK (v0.12.2)**, as for the starter.
**Run time:** about 15–20 minutes for everything. Sections 5 and 7 are the slow ones; lower `N_SEEDS` in the next cell to go faster (results will be noisier).

Everything is simulated (exact statevector): no jobs, no credits.

## 1. Setup

In [1]:
import importlib.util
missing = [m for m in ("qbraid", "qiskit") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Wrong kernel: pick qBraid-SDK (v0.12.2) at the top right, then run this cell again.")

import time, itertools
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

from check import (load_plants, load_demand, solve_milp, check_schedule, check_hour,
                   hour_ok, hour_cost, locks)

plants = load_plants()
demand, reserve = load_demand()
N = len(plants)
names = [p["plant_id"] for p in plants]
hours = np.arange(len(demand))
NEED = demand + reserve

milp_cost, milp_schedule, _, _ = solve_milp(plants, demand, reserve)

N_SEEDS = 10          # seeds per setting in the shot sweeps; 5 is faster but noisier
FIX_TODO2 = False     # see the note in section 2
print(f"{N} plants, {len(demand)} hours, MILP optimum ${milp_cost:,.0f}")

5 plants, 24 hours, MILP optimum $327,924


## 2. The starter's functions

Copied from `starter.ipynb` (sections 3–5), with one addition: `schedule_day_v2`, the same hour-by-hour loop with hooks for the experiments below (look-ahead credit, INTERP, penalty scaling, and an `"exact"` sampler that tries all 32 choices).

**Note on TODO 2.** The starter's `plant_hour_costs` computes `pmin * marginal - marginal_cheapest`, but the formula is `pmin * (marginal - marginal_cheapest)`. `FIX_TODO2 = False` keeps the starter's version so the numbers match the analysis. Set it to `True` to use the corrected formula. QAOA's statistics barely move either way (see section 5), and the exact-solver results don't depend on it, because the pick uses the exact `hour_cost`.

In [2]:
def plant_hour_costs(plants, was_on):
    m = min(p["marginal_cost_per_mwh"] for p in plants)
    c = np.zeros(len(plants))
    for i, p in enumerate(plants):
        extra = (p["pmin_mw"] * (p["marginal_cost_per_mwh"] - m) if FIX_TODO2
                 else p["pmin_mw"] * p["marginal_cost_per_mwh"] - m)
        c[i] = p["startup_cost"] * (1 - was_on[i]) + p["no_load_cost_per_h"] + extra
    return c


BLOCK_MW, N_SLACK = 50, 3
N_QUBITS = N + N_SLACK


def build_qubo(plants, demand_mw, reserve_mw, was_on, penalty=None):
    c = plant_hour_costs(plants, was_on)
    A = 2 * c.max() if penalty is None else penalty
    b = np.array([p["pmax_mw"] / BLOCK_MW for p in plants] + [-(2 ** k) for k in range(N_SLACK)])
    r = np.ceil((demand_mw + reserve_mw) / BLOCK_MW)
    Q = np.zeros((N_QUBITS, N_QUBITS))
    for i in range(len(c)):
        Q[i, i] += c[i]
    for j in range(N_QUBITS):
        Q[j, j] += A * (b[j] ** 2 - 2 * r * b[j])
        for k in range(j + 1, N_QUBITS):
            Q[j, k] += A * 2 * b[j] * b[k]
    return Q, A * r ** 2


INDEX_BITS = (np.arange(2 ** N_QUBITS)[:, None] >> np.arange(N_QUBITS)) & 1
PLANT_INDEX = INDEX_BITS[:, :N] @ (2 ** np.arange(N))
ALL_CHOICES = [list(c) for c in itertools.product([0, 1], repeat=N)]


def all_energies(Q, offset):
    return np.einsum("bj,jk,bk->b", INDEX_BITS, np.triu(Q), INDEX_BITS) + offset


def brute_force_qubo(Q, offset):
    E = all_energies(Q, offset)
    k = int(np.argmin(E))
    return INDEX_BITS[k], E[k]


def label(choice):
    on = [(choice >> i) & 1 for i in range(N)] if np.isscalar(choice) else choice
    return "+".join(n for n, u in zip(names, on) if u) or "none"


def qubo_to_ising(Q):
    h, J = np.zeros(len(Q)), {}
    for j in range(len(Q)):
        h[j] -= Q[j, j] / 2
        for k in range(j + 1, len(Q)):
            if Q[j, k] != 0:
                J[(j, k)] = Q[j, k] / 4
                h[j] -= Q[j, k] / 4
                h[k] -= Q[j, k] / 4
    return h, J


def qaoa_circuit(Q, p=1):
    h, J = qubo_to_ising(Q)
    scale = max(np.abs(h).max(), max(abs(v) for v in J.values()))
    gammas, betas = ParameterVector("gamma", p), ParameterVector("beta", p)
    qc = QuantumCircuit(len(Q))
    qc.h(range(len(Q)))
    for layer in range(p):
        for j, hj in enumerate(h):
            if hj != 0:
                qc.rz(2 * gammas[layer] * hj / scale, j)
        for (j, k), Jjk in J.items():
            qc.rzz(2 * gammas[layer] * Jjk / scale, j, k)
        qc.rx(2 * betas[layer], range(len(Q)))
    return qc, list(gammas) + list(betas)


def run_qaoa(Q, offset, p=1, start=None, maxiter=60):
    E = all_energies(Q, offset)
    E = (E - E.min()) / (E.max() - E.min())
    qc, params = qaoa_circuit(Q, p)

    def probs_at(angles):
        return Statevector(qc.assign_parameters(dict(zip(params, angles)))).probabilities()

    def average_energy(angles):
        return probs_at(angles) @ E

    if start is None or len(start) != 2 * p:
        grid = [(g, b) for g in np.linspace(0.1, 3.0, 15) for b in np.linspace(0.1, 1.5, 8)]
        g, b = min(grid, key=lambda gb: average_energy([gb[0]] * p + [gb[1]] * p))
        start = [g] * p + [b] * p
    result = minimize(average_energy, start, method="COBYLA", options={"maxiter": maxiter, "rhobeg": 0.2})
    return result.x, probs_at(result.x)


def choice_probs(probs):
    return np.bincount(PLANT_INDEX, weights=probs, minlength=2 ** N)


def add_locks(Q, offset, must_on, must_off, weight):
    Q = Q.copy()
    for i in must_on:
        Q[i, i] -= weight
        offset += weight
    for i in must_off:
        Q[i, i] += weight
    return Q, offset

In [3]:
def schedule_day_v2(demand=demand, reserve=reserve, credit_fn=None, credit_in_pick=True, p=1, shots=200,
                    sampler="qaoa", seed=7, init="warm", penalty_k=None, maxiter=60):
    # The starter's schedule_day, plus:
    #   credit_fn(t, was_on, demand, reserve) -> credit per plant   (look-ahead term, section 6)
    #   credit_in_pick: also use the credit when picking among the samples
    #   init="interp": INTERP angle initialization (section 7)
    #   penalty_k: penalty A = penalty_k * max(c) instead of the default 2 * max(c) (section 5)
    #   sampler="exact": try all 32 choices (a perfect per-hour solver, no QAOA)
    rng = np.random.default_rng(seed)
    history, log, angles, a1 = [], [], None, None
    was_on = [int(x["on_at_start"]) for x in plants]
    for t in range(len(demand)):
        credit = np.zeros(N) if credit_fn is None else credit_fn(t, was_on, demand, reserve)
        must_on, must_off = locks(plants, history)
        c = plant_hour_costs(plants, was_on)
        Q, offset = build_qubo(plants, demand[t], reserve[t], was_on,
                               penalty=None if penalty_k is None else penalty_k * c.max())
        Q = Q.copy()
        Q[np.arange(N), np.arange(N)] -= credit                    # the look-ahead term: E' = E - sum credit_i x_i
        Q, offset = add_locks(Q, offset, must_on, must_off, weight=2 * c.max())

        probs, samples = None, None
        if sampler == "qaoa":
            if init == "interp":
                angles, probs, a1 = run_qaoa_interp(Q, offset, p, start1=a1, maxiter=maxiter)
            else:                                                  # warm start from the previous hour
                angles, probs = run_qaoa(Q, offset, p=p, start=angles, maxiter=maxiter)
        elif sampler == "random":
            probs = np.full(2 ** N_QUBITS, 1 / 2 ** N_QUBITS)
        if probs is None:
            seen = ALL_CHOICES
        else:
            samples = rng.choice(2 ** N_QUBITS, size=shots, p=probs / probs.sum())
            seen = [list(INDEX_BITS[k][:N]) for k in np.unique(PLANT_INDEX[samples])]

        def score(ch):
            return hour_cost(plants, demand[t], ch, was_on) - (np.dot(credit, ch) if credit_in_pick else 0.0)

        ok = [ch for ch in seen if hour_ok(plants, demand[t], reserve[t], ch, must_on, must_off)]
        if not ok:                                                 # nothing fully valid: allow breaking a lock
            ok = [ch for ch in seen if hour_ok(plants, demand[t], reserve[t], ch)]
        if ok:
            pick = min(ok, key=score)
        else:
            pick = list(INDEX_BITS[np.bincount(PLANT_INDEX[samples]).argmax()][:N])

        # "best" = the choice minimizing the same score, over all 32 choices
        cands = [ch for ch in ALL_CHOICES if hour_ok(plants, demand[t], reserve[t], ch, must_on, must_off)]
        best = min(cands, key=score) if cands else None
        best_k = None if best is None else int(np.dot(best, 2 ** np.arange(N)))
        log.append(dict(
            hour=t, pick=pick, credit=credit.copy(), Q=Q, offset=offset, angles=angles, best_k=best_k,
            p_best=np.nan if probs is None or best_k is None else choice_probs(probs)[best_k],
            valid_share=np.nan if samples is None else np.mean(
                [hour_ok(plants, demand[t], reserve[t], INDEX_BITS[k][:N], must_on, must_off) for k in samples])))
        history.append(pick)
        was_on = pick
    return np.array(history), log


def day_gap(schedule, d=demand, r=reserve, mc=milp_cost):
    # (gap % vs the MILP, number of broken rules)
    res = check_schedule(plants, d, r, schedule, milp_cost=mc, verbose=False)
    return res["gap_pct"], len(res["violations"])


def shot_sweep(shot_list, samplers=("qaoa", "random"), seeds=None, **kw):
    # For each shot count and sampler: valid days out of `seeds`, and the average gap of the valid ones.
    seeds = N_SEEDS if seeds is None else seeds
    print(f"{'shots':>5}" + "".join(f"{s + ': valid, gap':>24}" for s in samplers))
    for shots in shot_list:
        cells = []
        for sm in samplers:
            gaps = [g for g, v in (day_gap(schedule_day_v2(shots=shots, sampler=sm, seed=sd, **kw)[0])
                                   for sd in range(seeds)) if v == 0]
            cells.append(f"{len(gaps):>2}/{seeds}, " + (f"{np.mean(gaps):6.2f}%" if gaps else "     -"))
        print(f"{shots:>5}" + "".join(f"{x:>24}" for x in cells))


# Sanity check: with no changes, this reproduces the starter's QAOA run.
s, _ = schedule_day_v2()
print("QAOA p=1, 200 shots, seed 7: gap %.2f%%, %d rules broken" % day_gap(s))

QAOA p=1, 200 shots, seed 7: gap 7.06%, 0 rules broken


## 3. Why does QAOA tie with random guessing?

QAOA and random guessing run through the same loop at 5–200 shots. Watch the last rows.

In [4]:
shot_sweep([5, 10, 20, 50, 200])

shots        qaoa: valid, gap      random: valid, gap
    5            0/10,      -            0/10,      -
   10           4/10,  13.33%            0/10,      -
   20          10/10,   8.42%           1/10,  10.75%
   50          10/10,   7.57%           9/10,   8.92%
  200          10/10,   7.06%          10/10,   7.13%


At 200 shots both give exactly the same day. Two reasons:

1. **200 samples cover the search space.** There are only 32 on/off choices, and 200 uniform samples hit about 31.9 of them on average.
2. **The classical step picks the answer.** The loop checks each sampled choice with `hour_ok` and keeps the cheapest by the exact `hour_cost`. If the best choice is sampled even once, it wins.

QAOA's advantage shows at low shot counts. Its samples are concentrated: about 2× random's chance of the best choice, and more valid samples:

In [5]:
_, qlog = schedule_day_v2(sampler="qaoa")
_, rlog = schedule_day_v2(sampler="random")
print(f"QAOA:   avg P(best) {np.nanmean([r['p_best'] for r in qlog]):.3f}, valid samples {np.mean([r['valid_share'] for r in qlog]):.0%}")
print(f"random: avg P(best) {1 / 2 ** N:.3f}, valid samples {np.mean([r['valid_share'] for r in rlog]):.0%}")

QAOA:   avg P(best) 0.064, valid samples 51%
random: avg P(best) 0.031, valid samples 29%


## 4. Why doesn't QAOA reach the MILP's minimum?

### 4a. The daily gap: a perfect per-hour solver has it too

`sampler="exact"` tries all 32 choices every hour, which is the best any per-hour sampler can do.

In [6]:
exact_sched, _ = schedule_day_v2(sampler="exact")
print("perfect per-hour solver: gap %.2f%%, %d rules broken\n" % day_gap(exact_sched))
print(f"{'hour':>4} {'need MW':>8}  {'hour-by-hour':<30} MILP")
for t in hours:
    if list(exact_sched[t]) != list(milp_schedule[t]):
        print(f"{t:>4} {NEED[t]:>8.0f}  {label(list(exact_sched[t])):<30} {label(list(milp_schedule[t]))}")

perfect per-hour solver: gap 7.06%, 0 rules broken

hour  need MW  hour-by-hour                   MILP
   5      364  COAL+OIL                       COAL+GAS_CC
   6      420  COAL+PEAKER_A+OIL              COAL+GAS_CC
   7      480  COAL+PEAKER_A+PEAKER_B+OIL     COAL+GAS_CC
   8      528  COAL+GAS_CC+PEAKER_A+PEAKER_B  COAL+GAS_CC
   9      577  COAL+GAS_CC+PEAKER_B           COAL+GAS_CC
  12      719  COAL+GAS_CC+OIL                COAL+GAS_CC+PEAKER_A
  13      745  COAL+GAS_CC+OIL                COAL+GAS_CC+PEAKER_A
  16      697  COAL+GAS_CC+PEAKER_A           COAL+GAS_CC


The gap is the hour-by-hour structure, not QAOA. In hour 5 the loop won't pay GAS_CC's $27,650 startup, and patches hours 5–7 with peakers and oil. From hour 8 GAS_CC is required anyway. The MILP sees the whole day and starts GAS_CC at hour 5.

### 4b. QAOA's own minimum: the penalty flattens the landscape

In [7]:
t = 12
Q12, off12 = build_qubo(plants, demand[t], reserve[t], milp_schedule[t - 1])
E = all_energies(Q12, off12)
En = (E - E.min()) / (E.max() - E.min())        # 0 = best bitstring, 1 = worst
g = int(np.argmin(E))
print(f"hour {t}: {int((En < 0.01).sum())} bitstrings within 1% of the minimum; "
      f"best vs 2nd best differ by {np.sort(En)[1] - np.sort(En)[0]:.4f} (normalized)\n")
print(f"{'p':>3} {'<E> (0=min)':>12} {'P(best bitstring)':>18} {'P(best choice)':>15}")
print(f"{'-':>3} {En.mean():>12.3f} {1 / 2 ** N_QUBITS:>18.3f} {1 / 2 ** N:>15.3f}   <- uniform (no QAOA)")
for p in [1, 2, 3, 4, 6]:
    _, pr = run_qaoa(Q12, off12, p=p, maxiter=300)
    print(f"{p:>3} {pr @ En:>12.3f} {pr[g]:>18.3f} {choice_probs(pr)[PLANT_INDEX[g]]:>15.3f}")

hour 12: 26 bitstrings within 1% of the minimum; best vs 2nd best differ by 0.0005 (normalized)

  p  <E> (0=min)  P(best bitstring)  P(best choice)
  -        0.270              0.004           0.031   <- uniform (no QAOA)
  1        0.046              0.026           0.085
  2        0.008              0.038           0.084
  3        0.072              0.002           0.018
  4        0.008              0.038           0.085
  6        0.008              0.037           0.092


QAOA gets the average energy close to the minimum, but the energy range is set by the penalty (millions of dollars for badly invalid bitstrings). The cost differences between valid choices (thousands of dollars) are tiny after rescaling, so many bitstrings look tied to QAOA. It learns "meet the capacity rule" much better than "be the cheapest". Note also that p=3 can do worse than p=1: the notebook's start copies the same angles into every layer (section 7 tries a better start).

## 5. Does lowering the penalty help?

`A = k × max(c)` each hour. The column "QUBO min = best" checks that the QUBO's lowest-energy bitstring is still the true best choice (MILP's previous hour as the starting state). Below some `k` the QUBO itself gives the wrong answer.

In [ ]:
start_state = [int(p["on_at_start"]) for p in plants]
print(f"{'A/max(c)':>8} {'QUBO min = best':>16} {'avg P(best)':>12} {'valid samples':>14} "
      f"{'20 shots: valid, gap':>22} {'10 shots: valid':>16}")
for k in [2, 1, 0.5, 0.25, 0.1, 0.05, 0.02]:
    m = 0
    for t in hours:
        was = milp_schedule[t - 1] if t > 0 else start_state
        Qt, ot = build_qubo(plants, demand[t], reserve[t], was, penalty=k * plant_hour_costs(plants, was).max())
        bits, _ = brute_force_qubo(Qt, ot)
        m += check_hour(plants, demand[t], reserve[t], bits[:N], was_on=was, verbose=False)["matches_best"]
    _, log = schedule_day_v2(penalty_k=k)
    res = {}
    for shots in [20, 10]:
        gaps = [g for g, v in (day_gap(schedule_day_v2(penalty_k=k, shots=shots, seed=sd)[0])
                               for sd in range(N_SEEDS)) if v == 0]
        res[shots] = (len(gaps), np.mean(gaps) if gaps else np.nan)
    print(f"{k:>8} {m:>13}/24 {np.nanmean([r['p_best'] for r in log]):>12.3f} "
          f"{np.mean([r['valid_share'] for r in log]):>14.0%} "
          f"{res[20][0]:>10}/{N_SEEDS}, {res[20][1]:6.2f}% {res[10][0]:>13}/{N_SEEDS}")

A/max(c)  QUBO min = best  avg P(best)  valid samples   20 shots: valid, gap  10 shots: valid


Down to about `0.5 × max(c)` the QUBO stays correct but P(best) barely moves. Below that the QUBO picks the wrong answer in many hours, so a higher P(best) means little. The penalty can't go low enough to matter. The pair terms are about `A × 2 × 7 × 7 ≈ 98A` (COAL and GAS_CC are 7 blocks each), and the circuit divides every angle by the largest coefficient, so the cost terms stay small.

## 6. Look-ahead QUBO term: crediting startups that will be paid anyway

From hour 8, demand + reserve exceeds 500 MW, which is everything except GAS_CC. So GAS_CC's startup gets paid regardless, and starting it earlier costs nothing extra in startup.

**The rule:** if some hour in the next `L` hours can't be covered without plant `i`, give it a credit:

$$E' = E - \sum_i \text{credit}_i\, x_i,\qquad \text{credit}_i = w \cdot \text{startup}_i \ \text{ if } \max_{t'\in t+1..t+L} \text{need}_{t'} > \text{fleet capacity} - \text{pmax}_i$$

It's a linear term, so only the diagonal of `Q` changes. The rule never names GAS_CC: on this fleet, it only ever fires for GAS_CC and COAL.

`symmetric=True` applies it whether the plant is on or off (turning it off means paying the startup again later). `symmetric=False` only credits plants that are off. That was the first version, and section 6d shows why it isn't enough.

In [ ]:
TOTAL_CAP = sum(p["pmax_mw"] for p in plants)


def necessity_credit(t, was_on, d, r, L=3, w=1.0, symmetric=True):
    future = (d + r)[t + 1 : t + 1 + L]
    credit = np.zeros(N)
    if len(future):
        for i, p in enumerate(plants):
            if (symmetric or not was_on[i]) and future.max() > TOTAL_CAP - p["pmax_mw"]:
                credit[i] = w * p["startup_cost"]
    return credit


def necessity(L=3, w=1.0, symmetric=True):
    return lambda t, was_on, d, r: necessity_credit(t, was_on, d, r, L, w, symmetric)

### 6a. Does the new QUBO help at all? Perfect per-hour solver, sweeping `L` and `w`

In [ ]:
W = [0.25, 0.5, 0.75, 1.0]
print(f"{'L':>3}" + "".join(f"{'w=' + str(w):>10}" for w in W))
for L in [1, 2, 3, 4, 6]:
    row = []
    for w in W:
        gp, v = day_gap(schedule_day_v2(sampler="exact", credit_fn=necessity(L, w))[0])
        row.append(f"{gp:>9.2f}%" + ("*" if v else ""))
    print(f"{L:>3}" + "".join(row))
print("(* = rules broken)")

In [ ]:
la_sched, la_log = schedule_day_v2(sampler="exact", credit_fn=necessity(3, 1.0))
print("L=3, w=1: gap %.2f%%\n" % day_gap(la_sched)[0])
print(f"{'hour':>4} {'need':>5}  {'no credit':<30} {'look-ahead':<28} MILP")
for t in hours:
    mark = "" if list(la_sched[t]) == list(milp_schedule[t]) else "   <- differs from MILP"
    print(f"{t:>4} {NEED[t]:>5.0f}  {label(list(exact_sched[t])):<30} {label(list(la_sched[t])):<28} "
          f"{label(list(milp_schedule[t]))}{mark}")

GAS_CC now starts at hour 5, as in the MILP. The remaining 1.18% is at hours 12–13 and 16: the loop uses OIL ($130 startup) where the MILP starts PEAKER_A ($5,150) for a 3-hour run. This rule can't see that, because PEAKER_A isn't strictly *needed* (section 8 tries a broader rule).

### 6b. What changes in the circuit

The circuit code is unchanged. `qubo_to_ising` maps `y = (1 − z)/2`, so subtracting `credit_i` from `Q[i,i]` shifts `h_i` by `+credit_i/2`, and `J` stays exactly the same. Only the `rz` angles on credited plant qubits change.

In [ ]:
t, was = 5, [int(v) for v in exact_sched[4]]
cr = necessity_credit(t, was, demand, reserve)
Q0, _ = build_qubo(plants, demand[t], reserve[t], was)
Q1 = Q0.copy()
Q1[np.arange(N), np.arange(N)] -= cr
h0, J0 = qubo_to_ising(Q0)
h1, J1 = qubo_to_ising(Q1)
print("hour 5 credit:", {n: float(c) for n, c in zip(names, cr)})
print("change in h:  ", np.round(h1 - h0, 1))
print("J unchanged:  ", J0 == J1)

### 6c. With QAOA, the credit has to be in the pick too

At 200 shots the classical pick decides the hour. If the credit is only in the QUBO, the pick (`min hour_cost`) still sees the full startup cost and ignores it.

In [ ]:
for cip in [False, True]:
    s, log = schedule_day_v2(credit_fn=necessity(3, 1.0), credit_in_pick=cip)
    gp, v = day_gap(s)
    print(f"credit in QUBO {'and in the pick' if cip else 'only           '}: gap {gp:.2f}%, {v} broken, "
          f"avg P(best) {np.nanmean([r['p_best'] for r in log]):.3f}")

### 6d. Symmetric vs off-only credit at low shots

When only off plants get the credit, low-shot runs sometimes switch COAL off at hours 5–7 (GAS_CC plus small plants look cheap) and restart it at hour 8, which breaks COAL's 48-hour min-down rule.

In [ ]:
for sym in [False, True]:
    fails = []
    for sd in range(N_SEEDS):
        s, _ = schedule_day_v2(credit_fn=necessity(3, 1.0, symmetric=sym), shots=20, seed=sd)
        res = check_schedule(plants, demand, reserve, s, milp_cost=milp_cost, verbose=False)
        if res["violations"]:
            fails.append(res["violations"][0])
    print(f"symmetric={sym}: {N_SEEDS - len(fails)}/{N_SEEDS} valid days at 20 shots"
          + (f"; e.g. {fails[0]}" if fails else ""))

### 6e. QAOA vs random with the look-ahead QUBO

In [ ]:
shot_sweep([10, 20, 50, 200], credit_fn=necessity(3, 1.0))

QAOA still reaches valid days with far fewer shots than random. At 20 shots the gap is still about 7%, because QAOA gives the right hour-5 choice only a small probability: the $27,650 credit is small next to the penalty terms (section 5).

In [ ]:
_, log = schedule_day_v2(credit_fn=necessity(3, 1.0))
print(f"hour 5: P(best choice) = {log[5]['p_best']:.3f}   (random: {1 / 2 ** N:.3f})")

## 7. INTERP angle initialization for p > 1

The notebook's `run_qaoa` grid-searches one (γ, β) and copies it into every layer. **INTERP** (Zhou et al., PRX 2020) grows the circuit one layer at a time. It tunes p layers, then builds a start for p+1 by padding the angle list with a zero at each end and blending neighbors:

$$\gamma'_i = \tfrac{i-1}{p}\gamma_{i-1} + \tfrac{p-i+1}{p}\gamma_i,\quad i = 1..p+1,\quad \gamma_0 = \gamma_{p+1} = 0$$

(and the same for β). This keeps the smooth ramp shape that good QAOA angles tend to have.

In [ ]:
def interp(v):
    p = len(v)
    v = np.concatenate([[0.0], v, [0.0]])
    return np.array([(i - 1) / p * v[i - 1] + (p - i + 1) / p * v[i] for i in range(1, p + 2)])


def run_qaoa_interp(Q, offset, p, start1=None, maxiter=60):
    # Tune p=1 (grid search, or warm start from start1), then add layers one at a time, seeding with INTERP.
    angles, probs = run_qaoa(Q, offset, p=1, start=start1, maxiter=maxiter)
    a1 = angles.copy()
    for q in range(2, p + 1):
        g, b = angles[: q - 1], angles[q - 1 :]
        angles, probs = run_qaoa(Q, offset, p=q, start=list(interp(g)) + list(interp(b)), maxiter=maxiter)
    return angles, probs, a1


print("example: gamma(p=2) = [0.4, 0.8]  ->  start for p=3:", interp(np.array([0.4, 0.8])))

### 7a. One hour at a time, from scratch (all 24 look-ahead QUBOs, COBYLA maxiter=200)

In [ ]:
_, la_qlog = schedule_day_v2(credit_fn=necessity(3, 1.0))


def scores(Q, off, probs, best_k):
    E = all_energies(Q, off)
    En = (E - E.min()) / (E.max() - E.min())
    return probs @ En, choice_probs(probs)[best_k]


print(f"{'p':>2} {'<E> const':>10} {'<E> INTERP':>11} {'P(best) const':>14} {'P(best) INTERP':>15} {'INTERP lower <E>':>17}")
for p in [1, 2, 3, 4, 5]:
    rc, ri = [], []
    for r in la_qlog:
        _, pc = run_qaoa(r["Q"], r["offset"], p=p, maxiter=200)
        _, pi, _ = run_qaoa_interp(r["Q"], r["offset"], p=p, maxiter=200)
        rc.append(scores(r["Q"], r["offset"], pc, r["best_k"]))
        ri.append(scores(r["Q"], r["offset"], pi, r["best_k"]))
    rc, ri = np.array(rc), np.array(ri)
    print(f"{p:>2} {rc[:, 0].mean():>10.4f} {ri[:, 0].mean():>11.4f} {rc[:, 1].mean():>14.3f} "
          f"{ri[:, 1].mean():>15.3f} {int((ri[:, 0] < rc[:, 0] - 1e-4).sum()):>11}/24 hrs")

From scratch, INTERP roughly doubles P(best) at p ≥ 3 and avoids the constant-angle start getting worse at p=3. Its average ⟨E⟩ is worse, pulled up by a few hours where it gets stuck. The two metrics disagree because ⟨E⟩ mostly measures validity (the penalty), while P(best) measures finding the cheapest valid choice.

### 7b. Over the whole day (slow: about 5–8 minutes with `N_SEEDS = 10`)

Inside the loop, the notebook already warm-starts each hour from the previous hour's angles. Is INTERP better than that?

In [ ]:
print(f"{'config':<22} {'avg P(best)':>11} {'time/day':>9} {'10 shots: valid, gap':>22} {'20 shots: valid, gap':>22}")
for name, p, init in [("p=1 (notebook)", 1, "warm"), ("p=4 warm start", 4, "warm"), ("p=4 INTERP", 4, "interp")]:
    t0 = time.perf_counter()
    _, log = schedule_day_v2(p=p, init=init, credit_fn=necessity(3, 1.0))
    dt = time.perf_counter() - t0
    cells = []
    for shots in [10, 20]:
        gaps = [g for g, v in (day_gap(schedule_day_v2(p=p, init=init, shots=shots, seed=sd,
                                                       credit_fn=necessity(3, 1.0))[0])
                               for sd in range(N_SEEDS)) if v == 0]
        cells.append(f"{len(gaps):>2}/{N_SEEDS}, " + (f"{np.mean(gaps):6.2f}%" if gaps else "     -"))
    print(f"{name:<22} {np.nanmean([r['p_best'] for r in log]):>11.3f} {dt:>8.1f}s {cells[0]:>22} {cells[1]:>22}")

In the day loop, the warm start does better than INTERP and is faster: neighboring hours have nearly the same QUBO, so the previous hour's p=4 angles are a better start than rebuilding from p=1. INTERP is the right tool for a single cold-started hour, such as the hardware run in the starter's section 7. For the whole day, keep the warm start and raise p.

## 8. Generalizing the look-ahead rule

The necessity rule only credits plants that are strictly needed. A broader rule, with no `w` to tune:

1. **Future plan:** for each hour, the cheapest valid choice if startups were free ("who you'd want running then").
2. **Credit:** if plant `i` is next in the plan `k` hours from now, `credit_i = max(0, startup_i − (k−1)·hold_i)`, where `hold_i` is the hourly cost of keeping it on. Keeping it on (or starting now) then wins exactly when `startup_i > k·hold_i`.

`L` optionally caps how far ahead it looks.

In [ ]:
def future_plan(d, r):
    plan = []
    for t in range(len(d)):
        ok = [c for c in ALL_CHOICES if hour_ok(plants, d[t], r[t], c)]
        plan.append(min(ok, key=lambda c: hour_cost(plants, d[t], c, c)) if ok else [0] * N)
    return np.array(plan)


m_cheapest = min(p["marginal_cost_per_mwh"] for p in plants)
HOLD = np.array([p["no_load_cost_per_h"] + p["pmin_mw"] * (p["marginal_cost_per_mwh"] - m_cheapest) for p in plants])


def plan_rule(d, r, L=None):
    plan = future_plan(d, r)

    def credit_fn(t, was_on, d_, r_):
        credit = np.zeros(N)
        for i, p in enumerate(plants):
            window = plan[t + 1 :, i] if L is None else plan[t + 1 : t + 1 + L, i]
            ahead = np.flatnonzero(window)
            if len(ahead):
                credit[i] = max(0.0, p["startup_cost"] - ahead[0] * HOLD[i])    # ahead[0] = k - 1
        return credit
    return credit_fn


plan = future_plan(demand, reserve)
print("future plan for a few hours:", {int(t): label(list(plan[t])) for t in [4, 5, 8, 12, 16, 23]})

### 8a. Five demand profiles (perfect per-hour solver; gap vs each profile's own MILP)

Profiles where no valid schedule exists, even for the MILP, are skipped.

In [ ]:
profiles = {"original": demand, "demand x0.9": demand * 0.9, "demand x1.08": demand * 1.08,
            "shift +4h": np.roll(demand, 4), "evening peak (reversed)": demand[::-1].copy(),
            "double peak": np.concatenate([demand[::2], demand[::2][::-1]])}
rules = {"no credit": lambda d, r: None,
         "necessity L=3": lambda d, r: necessity(3, 1.0),
         "plan (full day)": lambda d, r: plan_rule(d, r),
         "plan L=3": lambda d, r: plan_rule(d, r, L=3)}
print(f"{'profile':<24}" + "".join(f"{k:>17}" for k in rules))
totals = {k: [] for k in rules}
for name, d in profiles.items():
    d = np.round(d)
    r = np.round(0.1 * d)
    try:
        mc, _, _, _ = solve_milp(plants, d, r)
    except RuntimeError:
        print(f"{name:<24} (no valid schedule exists; skipped)")
        continue
    cells = []
    for k, make in rules.items():
        s, _ = schedule_day_v2(demand=d, reserve=r, sampler="exact", credit_fn=make(d, r))
        gp, v = day_gap(s, d, r, mc)
        totals[k].append(gp)
        cells.append(f"{gp:.2f}%" + ("*" if v else ""))
    print(f"{name:<24}" + "".join(f"{x:>17}" for x in cells))
print(f"{'average':<24}" + "".join(f"{np.mean(v):>16.2f}%" for v in totals.values()))
print("(* = rules broken)")

### 8b. The future-plan rule on the original day: where it differs from the MILP

In [ ]:
sp, _ = schedule_day_v2(sampler="exact", credit_fn=plan_rule(demand, reserve))
sn, _ = schedule_day_v2(sampler="exact", credit_fn=necessity(3, 1.0))
print(f"{'hour':>4}  {'plan (full day)':<30} {'necessity L=3':<28} MILP")
for t in hours:
    if list(sp[t]) != list(milp_schedule[t]) or list(sn[t]) != list(milp_schedule[t]):
        print(f"{t:>4}  {label(list(sp[t])):<30} {label(list(sn[t])):<28} {label(list(milp_schedule[t]))}")

- **Every rule cuts the gap by about 5–10×** on every profile, so the idea isn't tuned to one day.
- **The future-plan rule fixes hours 12–13** (PEAKER_A instead of OIL, as in the MILP). Over the full day it keeps PEAKER_A on through hour 22 because the plan wants it at hour 23; capping at L=3 fixes that.
- **It can over-credit:** "wanted" isn't "needed". At the peak it credits PEAKER_B when OIL is a cheaper substitute for 2 hours.
- **The necessity rule is the most robust on average** and the easiest to explain: it only credits a startup that truly can't be avoided.

## Summary

| Change | Effect |
|---|---|
| Starter as given | QAOA = random at 200 shots (7.06% gap); QAOA wins at 10–20 shots |
| Lower penalty | No real P(best) gain before the QUBO breaks (floor ≈ 0.5 × max c) |
| Look-ahead credit (necessity, L=3, w=1), in QUBO and pick | **7.06% → 1.18%**; still 10/10 valid days at 20 shots |
| INTERP, one hour from scratch | About 2× P(best) at p ≥ 3 |
| INTERP in the day loop | Warm start is better and faster; keep it |

**Next step beyond credits:** put 2–3 hours into one QUBO, with startup terms `x_{i,t}(1 − x_{i,t−1})` linking them. The look-ahead is then part of the problem instead of an approximation. That's about 16–24 qubits: simulable, but too deep for current hardware.